<a href="https://colab.research.google.com/github/deji4things2000/mlpro/blob/master/Byte_Pair_Encoding_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import numpy as np
import torch

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


Byte Pair Encoding

In [2]:
from collections import Counter

# ===========================
# 1. Load corpus (shakespeare.txt)
# ===========================
with open("/content/drive/MyDrive/shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Total characters in corpus:", len(text))

# ===========================
# 2. Initial vocabulary: unique characters
# ===========================
vocab = set(text)  # set of all unique chars
print("Initial vocab size (chars):", len(vocab))

# Represent corpus as a list of "words", each word is a list of chars + </w>
# This is a common BPE convention. [web:95][web:112]
words = text.split()
tokens = [list(w) + ["</w>"] for w in words]

# ===========================
# 3. Pair counting function
# ===========================
def get_pair_counts(tokens):
    counts = Counter()
    for word in tokens:
        for i in range(len(word) - 1):
            pair = (word[i], word[i+1])
            counts[pair] += 1
    return counts

# ===========================
# 4. Merge function (applies one pair)
# ===========================
def merge_tokens(tokens, pair):
    merged = []
    a, b = pair
    ab = a + b
    for word in tokens:
        i = 0
        new_word = []
        while i < len(word):
            if i < len(word) - 1 and word[i] == a and word[i+1] == b:
                new_word.append(ab)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        merged.append(new_word)
    return merged

# ===========================
# 5. BPE training loop
# ===========================
def learn_bpe(tokens, num_merges):
    merges = []          # list of (a,b) pairs in merge order
    vocab = set()        # grows as we add merged tokens

    # initialize vocab with all starting symbols
    for word in tokens:
        for tok in word:
            vocab.add(tok)

    for i in range(num_merges):
        pair_counts = get_pair_counts(tokens)
        if not pair_counts:
            print("No more pairs to merge at step", i)
            break

        # find most frequent pair
        best_pair, best_count = max(pair_counts.items(), key=lambda x: x[1])

        # stop if the best pair occurs only once (optional early stopping)
        if best_count < 2:
            print("Stopping early; highest freq is", best_count, "at step", i)
            break

        merges.append(best_pair)
        # add the merged symbol to vocab
        merged_symbol = best_pair[0] + best_pair[1]
        vocab.add(merged_symbol)

        # re-tokenize corpus by merging this pair everywhere
        tokens = merge_tokens(tokens, best_pair)

        if (i + 1) % 100 == 0:
            print(f"Completed {i+1} merges; current vocab size: {len(vocab)}")

    return merges, vocab, tokens

# ===========================
# 6. Run k merges
# ===========================
k = 1000  # number of merge steps requested
merges, final_vocab, final_tokens = learn_bpe(tokens, num_merges=k)

print("\nTotal merges learned:", len(merges))
print("Final vocab size:", len(final_vocab))

# Optionally inspect the top few merges
print("First 20 merges:")
for m in merges[:20]:
    print(m)

# ===========================
# 7. Helper: encode a new text using learned merges
# ===========================
def encode_word_bpe(word, merges):
    # start from characters + end-of-word marker
    tokens = list(word) + ["</w>"]
    for pair in merges:
        tokens = merge_tokens([tokens], pair)[0]
    return tokens

def encode_text_bpe(text, merges):
    encoded = []
    for w in text.split():
        encoded.extend(encode_word_bpe(w, merges))
    return encoded

# Example: encode a line after training (optional check)
example = "Alas, poor Yorick! I knew him, Horatio:"
encoded_example = encode_text_bpe(example, merges)
print("\nEncoded example:")
print(encoded_example)


Total characters in corpus: 1115394
Initial vocab size (chars): 65
Completed 100 merges; current vocab size: 164
Completed 200 merges; current vocab size: 264
Completed 300 merges; current vocab size: 364
Completed 400 merges; current vocab size: 464
Completed 500 merges; current vocab size: 564
Completed 600 merges; current vocab size: 664
Completed 700 merges; current vocab size: 764
Completed 800 merges; current vocab size: 864
Completed 900 merges; current vocab size: 964
Completed 1000 merges; current vocab size: 1064

Total merges learned: 1000
Final vocab size: 1064
First 20 merges:
('e', '</w>')
('t', 'h')
(',', '</w>')
('t', '</w>')
('s', '</w>')
('d', '</w>')
('o', 'u')
('e', 'r')
('y', '</w>')
('i', 'n')
(':', '</w>')
('a', 'n')
('o', 'r')
('o', '</w>')
('.', '</w>')
('e', 'n')
('a', 'r')
('o', 'n')
('l', 'l')
('h', 'a')

Encoded example:
['A', 'la', 's,</w>', 'poor</w>', 'Yor', 'i', 'ck', '!</w>', 'I</w>', 'k', 'ne', 'w</w>', 'him,</w>', 'H', 'or', 'a', 'ti', 'o', ':</w>']


In [3]:
print("Total merges learned:", len(merges))
for i, (a, b) in enumerate(merges):
    merged_symbol = a + b
    print(f"{i:4d}: {a!r} + {b!r} -> {merged_symbol!r}")


Total merges learned: 1000
   0: 'e' + '</w>' -> 'e</w>'
   1: 't' + 'h' -> 'th'
   2: ',' + '</w>' -> ',</w>'
   3: 't' + '</w>' -> 't</w>'
   4: 's' + '</w>' -> 's</w>'
   5: 'd' + '</w>' -> 'd</w>'
   6: 'o' + 'u' -> 'ou'
   7: 'e' + 'r' -> 'er'
   8: 'y' + '</w>' -> 'y</w>'
   9: 'i' + 'n' -> 'in'
  10: ':' + '</w>' -> ':</w>'
  11: 'a' + 'n' -> 'an'
  12: 'o' + 'r' -> 'or'
  13: 'o' + '</w>' -> 'o</w>'
  14: '.' + '</w>' -> '.</w>'
  15: 'e' + 'n' -> 'en'
  16: 'a' + 'r' -> 'ar'
  17: 'o' + 'n' -> 'on'
  18: 'l' + 'l' -> 'll'
  19: 'h' + 'a' -> 'ha'
  20: 'th' + 'e</w>' -> 'the</w>'
  21: 'i' + 's</w>' -> 'is</w>'
  22: 'f' + '</w>' -> 'f</w>'
  23: 'e' + 's' -> 'es'
  24: 'y' + 'ou' -> 'you'
  25: 'I' + '</w>' -> 'I</w>'
  26: 'll' + '</w>' -> 'll</w>'
  27: 't' + 'o</w>' -> 'to</w>'
  28: 'an' + 'd</w>' -> 'and</w>'
  29: 'o' + 'w' -> 'ow'
  30: 'e' + 'a' -> 'ea'
  31: 'e' + ',</w>' -> 'e,</w>'
  32: 'r' + '</w>' -> 'r</w>'
  33: 'in' + 'g' -> 'ing'
  34: 'w' + 'i' -> 'wi'
  35: